# Claar, Rembado et al. (2023) on dandiset 000458: EEG claims

**Paper.** Claar LD, Rembado I, Kuyat JR, Russo S, Marks LC, Olsen SR, Koch C (2023). Cortico-thalamo-cortical
interactions modulate electrically evoked EEG responses in mice. *eLife* 12:RP84630.
[doi:10.7554/eLife.84630](https://doi.org/10.7554/eLife.84630). The claims file was written from the version of
record (full text from Europe PMC), with its figure supplement legends.

**Dandiset.** [DANDI:000458](https://dandiarchive.org/dandiset/000458/0.230317.0039), version 0.230317.0039. The
paper's Data availability cites this version, and it is the only published version, so it is pinned. The draft
lists the same 24 assets with the same total size.

**What the dandiset holds.** 24 sessions of 23 mice: skull EEG (30 electrodes), Neuropixels LFP and sorted units in
18 sessions, wheel speed, isoflurane epochs, and every stimulus pulse with its current, target and state. This
notebook tests the EEG claims (Figures 1, 7C, 7D and 8); `claar_spikes.ipynb` tests the spiking claims.

| Claim | Here |
|---|---|
| stim-depths | blocked: the stimulation depth in mm is not in the dandiset |
| eeg-good-electrodes | tested |
| fig1d-two-erp-peaks | tested |
| fig1-supp1-example | tested |
| fig1e-erp-duration | tested |
| fig1f-erp-magnitude | tested |
| fig1-supp2-first-peak-flips | blocked: the stimulation site coordinates are not in the dandiset |
| state-trial-counts | tested |
| fig7c-erp-duration-state | tested |
| fig7d-erp-magnitude-state | tested |
| fig8b-pcist-awake-anesthesia | tested |
| fig8c-pcist-depth | tested |
| fig8d-pcist-three-states | tested |
| fig8e-pcist-recovery | tested |

Readings run for every claim (claims file, header): the stimulus intensity ("high", the largest of the session's
currents, and "medium", the middle one; sessions with one current give the same trials under both), and, where the
paper does not say, all awake trials or quiet trials only. Only trials the file marks `is_valid` are used.

In [ ]:
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import claar

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
urls = claar.asset_urls()
metas = [claar.read_meta(p, u) for p, u in urls]
print(f"{len(metas)} sessions from dandiset {claar.DANDISET_ID} version {claar.DANDISET_VERSION}, metadata read in {time.time() - started:.0f} s")

## Data check

In [ ]:
rows = []
for m in metas:
    el = m["estim"]
    eeg_el = m["electrodes"].query("group_name == 'EEG array'")
    rows.append({"subject": m["subject"], "session": m["session"], "stimulus note": m["stimulus"].replace("single pulse electrical stimuli targeted to ", ""),
                 "sites": ", ".join(sorted(el.site.unique())), "EEG good": int(eeg_el.is_data_valid.sum()),
                 "EEG hours": round(m["eeg_shape"][0] / claar.FS_EEG / 3600, 2), "probes": len(m["probes"]),
                 "units": 0 if m["units"] is None else len(m["units"]),
                 "pulses": len(el), "valid": int(el.is_valid.sum()),
                 "visual trials": int((m["trials"].stimulus_type == "visual").sum()),
                 "epochs": "; ".join(f"{r.tag} {r.start:.0f}-{r.stop:.0f} s" for r in m["epochs"].itertuples())})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("\nsubjects:", info.subject.nunique(), " sessions:", len(info))
sites = [(i, s) for i, m in enumerate(metas) for s in sorted(m["estim"].site.unique())]
print("stimulation sites (session x target):", len(sites))
no_valid = [(metas[i]["subject"], s) for i, s in sites if not metas[i]["estim"].query("site == @s").is_valid.any()]
print("sites whose pulses are all marked invalid (left out of the analyses):", no_valid)
print(pd.Series([s for _, s in sites]).value_counts().to_string())
print("\nTrial-table columns:", list(metas[0]["trials"].columns))
print("Values of estim_target_depth:", sorted({d for m in metas for d in m["estim"].estim_target_depth.unique()}))
print("\nElectrode-table columns:", metas[0]["electrode_columns"])
eeg_xyz = pd.concat([m["electrodes"].query("group_name == 'EEG array'")[["x", "y", "z"]] for m in metas])
print("EEG electrode x, y, z over sessions: ", {c: (round(float(eeg_xyz[c].min()), 2), round(float(eeg_xyz[c].max()), 2)) for c in "xyz"},
      "; distinct (x, y) positions:", len(eeg_xyz[["x", "y"]].drop_duplicates()))
probe_loc = pd.concat([m["electrodes"].query("group_name != 'EEG array'") for m in metas])
print("probe channels:", len(probe_loc), "; their location values (CCF acronyms) include:", sorted(probe_loc.location.unique())[:40])

The dandiset holds 23 subjects and 24 sessions (subject 569073 has two, MOs and SSp), matching the 23 included
mice of the Methods. There are 31 stimulation sites, 13 superficial and 18 deep, the N of Figure 8B (31 sessions)
and 8C (13 and 18). The trial table gives the target depth only as "superficial" or "deep"; the depth of the
stimulating electrode in mm, and its coordinates, are not in the files.

In [ ]:
# currents, stage sizes and states per site
rows = []
for i, s in sites:
    el = metas[i]["estim"].query("site == @s")
    lv = claar.intensity_levels(el.current)
    rows.append({"subject": metas[i]["subject"], "site": s, "currents uA": "/".join(str(int(c)) for c in sorted(el.current.unique())),
                 "high": lv["high"], "medium": lv["medium"],
                 **{st: int((el.state == st).sum()) for st in ["quiet", "running", "isoflurane", "recovery1", "recovery2", "recovery3"]},
                 "invalid": int((~el.is_valid).sum())})
site_table = pd.DataFrame(rows)
print(site_table.to_string(index=False))

# running speed and the file's is_running flag
aw = pd.concat([m["estim"].query("behavioral_epoch == 'awake'") for m in metas])
print("\nwheel speed unit:", metas[0]["speed_unit"], "; range over sessions:",
      round(min(m["speed"].min() for m in metas), 2), "to", round(max(m["speed"].max() for m in metas), 1))
print("awake pulses:", len(aw), "; is_running agrees with speed > 0 somewhere in -0.5 to +0.5 s:",
      round((aw.is_running == (aw.speed_max > 0)).mean(), 4), "; disagreements:", int((aw.is_running != (aw.speed_max > 0)).sum()))
print("units' waveform_duration range (file says s):",
      round(min(m["units"].waveform_duration.min() for m in metas if m["units"] is not None), 3), "to",
      round(max(m["units"].waveform_duration.max() for m in metas if m["units"] is not None), 3))

Each stage has 120 pulses per current in most sessions (three currents, as in the Methods); five early superficial
sessions differ (one or three currents, 60 to 300 pulses). The recovery stages split at 30 min after the end of
the isoflurane epoch give a first and, in 11 sites, a second stage. The file's `is_running` agrees with the wheel
speed by the paper's rule. The wheel speed is in cm/s and never negative, as described.

In [ ]:
# Clock check: the stimulus artifact in the raw EEG must sit at the trial onsets of the trial table.
# Contrast: raw amplitude at true onsets against onsets shifted by 1 s.
m = metas[[k for k, mm in enumerate(metas) if mm["subject"] == "571619"][0]]
f = claar.open_nwb(m["url"])
ts = f["acquisition/ElectricalSeriesEEG/timestamps"][:]
d = f["acquisition/ElectricalSeriesEEG/data"]
print("EEG timestamps: first", round(ts[0], 3), "s, median interval", round(np.median(np.diff(ts)) * 1e3, 4), "ms ->",
      round(1 / np.median(np.diff(ts))), "Hz; conversion", m["eeg_conversion"], "V per unit")
on = m["estim"].start_time.values[::20]
amp = []
for shift in (0.0, 1.0):
    a = []
    for t in on:
        k = np.searchsorted(ts, t + shift)
        x = d[k - 25:k + 25, :].astype(float) * m["eeg_conversion"] * 1e6
        x -= x[:20].mean(0)
        a.append(np.abs(x).max(1))
    amp.append(np.median(a, axis=0))
print("median over pulses and electrodes-max of |raw EEG| (uV) from -10 ms to +10 ms, true onsets:")
print(np.round(amp[0]).astype(int))
print("onsets shifted by +1 s:")
print(np.round(amp[1]).astype(int))
f.close()

In this session the artifact spans indices 24 and 25 of the window (656 and 696 uV), that is k - 1 and k, where k is the first EEG sample at or after the trial onset (index 25), and it is absent one second later, so the EEG and the trial table share a clock. The mask therefore starts at k - 1 (`claar.MASK_START`): samples k - 1 to k + 4 are replaced by k - 7 to k - 2, reversed. The same profile is printed below for every site, from the pulses of each site as it is read. The wheel speed's clock (starting at 0 s, 100 Hz) is checked by the state contrast of Figure 7, where running trials give a different ERP.

## EEG of every site

In [ ]:
sites = [(i, s) for i, s in sites if (metas[i]["subject"], s) not in no_valid]
def summary(args):
    i, s = args
    return claar.eeg_site_summary(metas[i], s)

t0 = time.time()
with ThreadPoolExecutor(max_workers=2) as ex:
    res = list(ex.map(summary, sites))
print(f"{len(res)} sites summarised in {time.time() - t0:.0f} s")
for r in res:
    r["depth"] = r["site"].split("-")[1]
    r["region"] = r["site"].split("-")[0]
ex_site = [k for k, r in enumerate(res) if r["subject"] == "571619" and r["site"] == "MOs-deep"][0]
chk = [claar.pcist_check(r["erp"][("high", "awake")], r["times"]) for r in res]
diff = np.array([abs(a - b) for a, b in chk])
print(f"PCIst of the fast count against the original code on decimated ERPs (step 10), all {len(chk)} sites: "
      f"largest absolute difference {diff.max():.2e}; values {min(min(x) for x in chk):.2f} to {max(max(x) for x in chk):.2f}")

Artifact profile of every site (median over its pulses of the largest raw |EEG| over good electrodes, uV, relative to the mean of k - 25 to k - 11), and what remains of it in the trial-averaged ERP: the largest |z| of the z-scored GFP from -20 to -2 ms and from +2 to +20 ms (all awake trials, high current). A residue of the artifact would show as a peak just before the onset.

In [ ]:
rows = []
for r in res:
    a = r["artifact"]
    z = claar.gfp_z(r["erp"][("high", "awake")], r["times"])
    t = r["times"]
    rows.append({"subject": r["subject"], "site": r["site"], "peak at k +": int(np.argmax(a)) - 10,
                 **{f"k{d:+d}": int(round(a[10 + d])) for d in (-3, -2, -1, 0, 1, 2, 5, 10)},
                 "max |z| -20..-2 ms": round(float(np.abs(z[(t >= -0.020) & (t <= -0.002)]).max()), 2),
                 "max z +2..+20 ms": round(float(z[(t >= 0.002) & (t <= 0.020)].max()), 2)})
art = pd.DataFrame(rows)
print(art.to_string(index=False))
print("\nsites whose artifact peaks at k - 1:", int((art["peak at k +"] == -1).sum()), "of", len(art),
      "; at k:", int((art["peak at k +"] == 0).sum()))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
t = res[0]["times"]
w = (t >= -0.020) & (t <= 0.020)
zs = np.array([claar.gfp_z(r["erp"][("high", "awake")], t)[w] for r in res])
for k, dep in enumerate(("superficial", "deep")):
    sel = [j for j, r in enumerate(res) if r["depth"] == dep]
    ax[k].plot(1e3 * t[w], zs[sel].T, lw=0.6, color="0.5")
    ax[k].axvspan(-0.4, 1.6, color="r", alpha=0.15)
    ax[k].set(title=f"{dep}: z-scored GFP, high current, all awake", xlabel="ms", ylabel="z")
plt.tight_layout()
print("z range plotted:", round(float(zs.min()), 1), "to", round(float(zs.max()), 1), "(no points outside the axes)")

The artifact is largest at k - 1 in 24 of 30 sites and at k in 6, and in every site it is far above the pre-pulse level at both samples, so the EEG of every session shares the trial table's clock and the mask covers the artifact peak. In several sites the EEG stays above its pre-pulse level after the masked samples (for example 586466 SSp-deep, 129 uV at k + 5 against 62 at k - 3; 551397 MOs-deep, 187 uV at k + 10 against 50), and the zero-phase filter spreads this step to both sides of the onset: the z-scored GFP reaches |z| = 21.78 between -20 and -2 ms in the worst site (590480 SSp-superficial) and exceeds 10 in 12 of 30 sites. After the onset this residue is not separated from the response. The -20 to 0 ms window lies inside the baselines of the GFP z-score (-2 to 0 s) and of PCIst (-0.8 to -0.002 s), so the residue raises their SD. Its effect is measured by repeating the GFP metrics, and PCIst at the medium current, with baselines that end 20 ms before the pulse (rows "-2 to -0.02 s" and matchings ending in "-excl20" below). This variant is outside the claims file's criteria and decides no outcome.

In [ ]:
def metric_table(level, state, matching="all"):
    rows = []
    for r in res:
        key = (level, state, matching)
        if key not in r["metrics"]:
            continue
        rows.append({"subject": r["subject"], "site": r["site"], "depth": r["depth"], "region": r["region"],
                     "n": r["n"][(level, state)], **r["metrics"][key]})
    return pd.DataFrame(rows)

def mean_sem(x):
    x = np.asarray(x, float)
    return f"{x.mean():.2f} +/- {stats.sem(x):.2f} (n={len(x)})"

## eeg-good-electrodes

> After artifact masking and visual inspection, an average of 23 +/- 4 of the 30 EEG electrodes per subject were
> kept as artifact-free.

In [ ]:
good = info.groupby("subject")["EEG good"].mean()
print("good (is_data_valid) EEG electrodes per session:", info["EEG good"].tolist())
print(f"per subject (23 subjects, the two sessions of 569073 averaged): mean {good.mean():.2f}, SD {good.std():.2f}, range {good.min()}-{good.max()}")
print(f"per session (24): mean {info['EEG good'].mean():.2f}, SD {info['EEG good'].std():.2f}")
print("expected 23 +/- 4; within tolerance:", abs(good.mean() - 23) <= 4)

**Result: reproduced.** The files mark good EEG electrodes (`is_data_valid`); their mean per subject is 23.07 (SD 4.01, range 17 to 30) against 23 +/- 4. That these marks are the paper's visual selection is consistent with the data, not stated in the files.

## fig1-supp1-example

> In the example deep MOs session, the ERP lasts 1.01 s and has a magnitude of 11.13 SD.s.

The example is "the deep MOs subject of Figure 1C and 1D (bottom left)", whose superficial MOs stimulation is shown
top left "in the same subject" (Figure 1D legend). Subject 571619 is the only one with both superficial and deep
MOs stimulation, so it is the example.

In [ ]:
r = res[ex_site]
print("example:", r["subject"], r["site"])
rows = []
for lvl in claar.LEVELS:
    for st in ("awake", "quiet"):
        mt = r["metrics"][(lvl, st, "all")]
        rows.append({"intensity": lvl, "trials": st, "n": r["n"][(lvl, st)], **{k: round(float(v), 3) for k, v in mt.items()}})
ex_tab = pd.DataFrame(rows)
print(ex_tab.to_string(index=False))
print("currents (uA):", site_table.query("subject == '571619' and site == 'MOs-deep'")[["high", "medium"]].to_dict("records"))
ok = ex_tab[(abs(ex_tab.duration_a - 1.01) <= 0.101) | (abs(ex_tab.duration_b - 1.01) <= 0.101)]
ok = ok[(abs(ok.magnitude_a - 11.13) <= 1.113) | (abs(ok.magnitude_b - 11.13) <= 1.113)]
print("\nreadings with both values within tolerance (duration 1.01 +/- 0.101 s, magnitude 11.13 +/- 1.113 SD.s):")
print(ok[["intensity", "trials", "duration_a", "duration_b", "magnitude_a", "magnitude_b"]].to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
t = r["times"]
w = (t >= -0.3) & (t <= 1.1)
erp = r["erp"][("high", "awake")]
ax[0].plot(t[w], erp[:, w].T, color="k", lw=0.5)
ax[0].set(xlabel="time from pulse (s)", ylabel="uV", title="571619 deep MOs, awake, high current")
z = claar.gfp_z(erp, t)
ax[1].plot(t[w], z[w], color="m")
ax[1].axhline(3, ls="--", color="k")
ax[1].set(xlabel="time from pulse (s)", ylabel="GFP z")
plt.tight_layout()
print("plotted ranges: ERP", round(float(erp[:, w].min()), 1), "to", round(float(erp[:, w].max()), 1), "uV; GFP z", round(float(z[w].min()), 1), "to", round(float(z[w].max()), 1))

**Result: reproduced.** In subject 571619, deep MOs, all awake trials at the medium current (40 uA) give a duration of 1.011 s and a magnitude of 11.133 SD.s, against 1.01 s and 11.13 SD.s, with duration and magnitude read as the total time and the area above z = 3. This is the only reading with both values within tolerance; the high current gives 1.169 s and 12.556 SD.s, both above.

## fig1-supp2-first-peak-flips (blocked)

> The amplitude of the first ERP peak decays with distance from the stimulation site and eventually flips sign.

The criterion needs the distance of each electrode from the stimulation site. The EEG electrodes have coordinates (data check), the stimulating electrode has none in the files, so the claim is blocked. What can be shown without that distance is whether both signs occur at the time of the early GFP peak; this does not test the claim.

In [ ]:
rows = []
for r in res:
    erp, t = r["erp"][("high", "awake")], r["times"]
    z = claar.gfp_z(erp, t)
    w = np.where((t >= 0.002) & (t <= 0.050))[0]
    j = w[np.argmax(z[w])]
    v = erp[:, j]
    rows.append({"subject": r["subject"], "site": r["site"], "peak ms": round(1e3 * t[j], 1), "electrodes": len(v),
                 "positive": int((v > 0).sum()), "negative": int((v < 0).sum())})
sg = pd.DataFrame(rows)
print(sg.to_string(index=False))
print("\nsites with both signs at the early GFP peak:", int(((sg.positive > 0) & (sg.negative > 0)).sum()), "of", len(sg),
      "(the ERP is average-referenced, so both signs are expected at any time point)")

## fig1d-two-erp-peaks

> Deep (layer 5/6) stimulation in the awake state usually evokes an ERP with two prominent peaks, near 25 ms and near
> 180 ms; the late component is not evident with superficial stimulation, in MOs or SSp.

Readings (claims file): (a) a local maximum of the z-scored GFP above 3 in 2 to 50 ms and another in 150 to 300 ms;
(b) the same with 2 to 25 ms and 100 to 300 ms.

In [ ]:
rows = []
for lvl in claar.LEVELS:
    for st in ("awake", "quiet"):
        for rd in ("a", "b"):
            d = pd.DataFrame([{"depth": r["depth"], "region": r["region"], **r["peaks"][(lvl, st)][rd]} for r in res if (lvl, st) in r["peaks"]])
            deep, sup = d[d.depth == "deep"], d[d.depth == "superficial"]
            rows.append({"intensity": lvl, "trials": st, "reading": rd,
                         "deep both peaks": f"{int((deep.early & deep.late).sum())}/{len(deep)}",
                         "superficial no late": f"{int((~sup.late).sum())}/{len(sup)}",
                         "deep MOs both": f"{int((deep[deep.region=='MOs'].early & deep[deep.region=='MOs'].late).sum())}/{(deep.region=='MOs').sum()}",
                         "deep SSp both": f"{int((deep[deep.region=='SSp'].early & deep[deep.region=='SSp'].late).sum())}/{(deep.region=='SSp').sum()}",
                         "sup MOs no late": f"{int((~sup[sup.region=='MOs'].late).sum())}/{(sup.region=='MOs').sum()}",
                         "sup SSp no late": f"{int((~sup[sup.region=='SSp'].late).sum())}/{(sup.region=='SSp').sum()}",
                         "holds": ((deep.early & deep.late).mean() > 0.5) and ((~sup.late).mean() > 0.5)})
peaks_tab = pd.DataFrame(rows)
print(peaks_tab.to_string(index=False))
d = pd.DataFrame([{"subject": r["subject"], "site": r["site"], **{k: round(v, 3) if isinstance(v, float) else v for k, v in r["peaks"][("high", "awake")]["a"].items()}} for r in res])
print("\nper site, high current, all awake trials, reading (a); late_max is the largest GFP z in 150-300 ms:")
print(d.to_string(index=False))

**Result: not reproduced.** Every deep site (18 of 18) has GFP peaks above z = 3 in both the early and the late window under every reading, but the late peak is also present at most superficial sites: only 0 to 2 of the 12 superficial sites lack it, under every reading. The superficial late peaks are smaller (largest late z in the per-site table) but above the threshold of the criterion.

## fig1e-erp-duration

> ERPs evoked by superficial stimulation were shorter than those evoked by deep stimulation (0.4 +/- 0.0 against
> 0.5 +/- 0.1 s), but the difference was not significant.

Per site, awake ERP; duration readings (a) total time above z = 3, (b) onset to the last sample above z = 3.
Claim of absence, judged on the paper's test (scipy `ttest_ind`, two-sided); the paper's N is 12 superficial and 18
deep, the dandiset has 13 and 18 sites.

In [ ]:
rows = []
for lvl in claar.LEVELS:
    for st, base in (("awake", "all"), ("quiet", "all"), ("awake", "excl20"), ("quiet", "excl20")):
        d = metric_table(lvl, st, base)
        for rd in ("duration_a", "duration_b"):
            s, dp = d[d.depth == "superficial"][rd], d[d.depth == "deep"][rd]
            tt = stats.ttest_ind(s, dp)
            rows.append({"intensity": lvl, "trials": st, "z baseline": "-2 to 0 s" if base == "all" else "-2 to -0.02 s", "reading": rd, "superficial": mean_sem(s), "deep": mean_sem(dp),
                         "t": round(tt.statistic, 2), "p": round(tt.pvalue, 4),
                         "within tol": bool(abs(s.mean() - 0.4) <= 0.05 and abs(dp.mean() - 0.5) <= 0.1),
                         "p>=0.05 and sup<deep": bool(tt.pvalue >= 0.05 and s.mean() < dp.mean())})
dur_tab = pd.DataFrame(rows)
print(dur_tab.to_string(index=False))
d = metric_table("high", "awake")
print("\nper site, high current, all awake trials:")
print(d[["subject", "site", "n", "duration_a", "duration_b", "magnitude_a", "magnitude_b"]].round(3).to_string(index=False))

**Result: partially reproduced.** Superficial ERPs are shorter than deep ERPs and the difference is not significant under every reading of duration (a). At the medium current with all awake trials, the reading that matches the example of fig1-supp1, superficial ERPs last 0.33 +/- 0.04 s and deep ERPs 0.49 +/- 0.06 s (n = 12 and 18), p = 0.081 (paper p = 0.0843); the deep mean is within tolerance, the superficial mean is 0.02 s below 0.4 +/- 0.05. At the high current both means are above tolerance (0.56 and 0.65 s, p = 0.53). No reading of the criterion puts both means within tolerance. With the baseline ending 20 ms before the pulse (the artifact check above), the medium current gives 0.37 +/- 0.04 and 0.50 +/- 0.06 s, p = 0.151 (quiet trials 0.37 and 0.53 s, p = 0.092), both within tolerance; the outcome on the criterion's -2 to 0 s baseline therefore turns on the artifact residue in the last 20 ms of the baseline.

## fig1f-erp-magnitude

> ERPs evoked by superficial stimulation had a significantly smaller normalized magnitude than those evoked by deep
> stimulation (3.3 +/- 0.8 against 6.2 +/- 0.9 SD.s).

Readings (a) the area above z = 3 (SD.s); (b) that area divided by the area under the baseline. The area under the signed baseline z-score is zero by construction (printed as `baseline_signed_area`), so (b) cannot be computed as written. The table also shows (b) with the area under |z| in the baseline as divisor, for comparison only: it is close to a rescaling of (a), and the paper prints the magnitude in SD.s, the unit of (a). The outcome rests on (a).

In [ ]:
rows = []
for lvl in claar.LEVELS:
    for st, base in (("awake", "all"), ("quiet", "all"), ("awake", "excl20"), ("quiet", "excl20")):
        d = metric_table(lvl, st, base)
        for rd in ("magnitude_a", "magnitude_b"):
            s, dp = d[d.depth == "superficial"][rd], d[d.depth == "deep"][rd]
            tt = stats.ttest_ind(s, dp)
            rows.append({"intensity": lvl, "trials": st, "z baseline": "-2 to 0 s" if base == "all" else "-2 to -0.02 s", "reading": rd, "superficial": mean_sem(s), "deep": mean_sem(dp),
                         "t": round(tt.statistic, 2), "p": round(tt.pvalue, 4),
                         "within tol": bool(abs(s.mean() - 3.3) <= 0.8 and abs(dp.mean() - 6.2) <= 0.9),
                         "p<0.05 and sup<deep": bool(tt.pvalue < 0.05 and s.mean() < dp.mean())})
mag_tab = pd.DataFrame(rows)
print(mag_tab.to_string(index=False))
print("max |baseline_signed_area| over sites:", float(np.max(np.abs(metric_table('high', 'awake').baseline_signed_area))))

**Result: reproduced.** With reading (a), the area above z = 3 in SD.s, the medium current gives a superficial magnitude of 3.03 +/- 0.78 and a deep magnitude of 5.86 +/- 0.90 with all awake trials, both within tolerance, t-test p = 0.035 (paper 3.3 +/- 0.8, 6.2 +/- 0.9, p = 0.0354); quiet trials give 2.85 and 5.76, p = 0.028. At the high current the means are above tolerance (5.70 and 9.31, p = 0.069 with all awake trials). With the baseline ending 20 ms before the pulse (variant, decides nothing) the medium current gives 3.38 and 6.22, p = 0.047, also within tolerance.

## state-trial-counts

> Per animal, the state comparisons averaged a median of 97 quiet-wakefulness, 21 running and 120 anesthetized
> trials, across 17 animals.

Session sets (claims file): (a) the deep sites, one value per animal (the two deep sites of 569073 averaged);
(b) every site, one value per animal (sites of one animal averaged).

In [ ]:
def per_animal(df, cols):
    return df.groupby("subject")[cols].mean()

rows = []
for lvl in claar.LEVELS:
    d = pd.DataFrame([{"subject": r["subject"], "depth": r["depth"], "quiet": r["n"][(lvl, "quiet")],
                       "running": r["n"][(lvl, "running")], "isoflurane": r["n"][(lvl, "isoflurane")]} for r in res])
    for name, sub in (("a: deep sites", d[d.depth == "deep"]), ("b: all sites", d)):
        pa = per_animal(sub, ["quiet", "running", "isoflurane"])
        q = pa.quantile([0.25, 0.5, 0.75])
        rows.append({"intensity": lvl, "set": name, "animals": len(pa),
                     **{c: f"{q.loc[0.5, c]:.1f} [{q.loc[0.25, c]:.1f}-{q.loc[0.75, c]:.1f}]" for c in pa.columns},
                     "within tol": bool(abs(q.loc[0.5, "quiet"] - 97) <= 9.7 and abs(q.loc[0.5, "running"] - 21) <= 2.1 and abs(q.loc[0.5, "isoflurane"] - 120) <= 12)})
counts_tab = pd.DataFrame(rows)
print(counts_tab.to_string(index=False))
print("\nvalid trials only; the paper's medians are 97 [86-104], 21 [16-27], 120 [120-120] over 17 animals")

**Result: reproduced.** Over the 17 animals with deep stimulation, the median counts per animal are 99, 21 and 120 at the high current and 97, 21 and 120 at the medium current, against 97, 21 and 120. Over all 23 animals the running median is 24, out of tolerance.

## fig7c-erp-duration-state and fig7d-erp-magnitude-state

> ERP duration decreased from quiet wakefulness (0.5 +/- 0.1 s) to running (0.3 +/- 0.0 s) and anesthesia
> (0.3 +/- 0.0 s); the late component is diminished when running and absent under anesthesia.

> ERP magnitude decreased from quiet wakefulness (6.1 +/- 0.9) to running (1.6 +/- 0.3) and anesthesia (1.6 +/- 0.3).

Per animal (session sets as above), states unmatched or matched by random subsampling to the smallest state;
Friedman test (scipy `friedmanchisquare`).

In [ ]:
exp = {"duration": {"quiet": (0.5, 0.1), "running": (0.3, 0.05), "isoflurane": (0.3, 0.05)},
       "magnitude": {"quiet": (6.1, 0.9), "running": (1.6, 0.3), "isoflurane": (1.6, 0.3)}}
state_rows = {"duration": [], "magnitude": []}
for lvl in claar.LEVELS:
    for matching in ("all", "matched", "excl20"):
        tabs = []
        for st in ("quiet", "running", "isoflurane"):
            d = metric_table(lvl, st, matching)
            d["state"] = st
            tabs.append(d)
        d = pd.concat(tabs)
        for setname, sub in (("a: deep", d[d.depth == "deep"]), ("b: all", d)):
            for measure in ("duration", "magnitude"):
                for rd in ("a", "b"):
                    col = f"{measure}_{rd}"
                    w = sub.pivot_table(index="subject", columns="state", values=col, aggfunc="mean").dropna()
                    fr = stats.friedmanchisquare(w.quiet, w.running, w.isoflurane)
                    within = all(abs(w[s].mean() - exp[measure][s][0]) <= exp[measure][s][1] for s in exp[measure])
                    top = (w.quiet.mean() > w.running.mean()) and (w.quiet.mean() > w.isoflurane.mean())
                    state_rows[measure].append({"intensity": lvl, "matching": matching, "set": setname, "reading": rd, "animals": len(w),
                                                **{s: mean_sem(w[s]) for s in ("quiet", "running", "isoflurane")},
                                                "Q": round(fr.statistic, 1), "p": f"{fr.pvalue:.2g}", "within tol": within,
                                                "quiet largest": top, "all hold": bool(within and fr.pvalue < 0.05 and top)})
dstate = pd.DataFrame(state_rows["duration"])
print(dstate.to_string(index=False))

**Result: partially reproduced.** Duration falls from quiet wakefulness to running and isoflurane in every reading except reading (b) at the medium current, and the Friedman test is significant in most (17 deep animals, medium current, unmatched, reading a: 0.52, 0.25 and 0.22 s, Q = 20.1, p = 4.3e-05; paper Q = 15.6). No reading puts the three means within tolerance: at the medium current the isoflurane mean (0.22 s) is below 0.3 +/- 0.05, at the high current the quiet mean (0.65 s) is above 0.5 +/- 0.1, and matched trials shorten the quiet mean to 0.31 to 0.39 s. With the baseline ending 20 ms before the pulse (variant) the medium current gives 0.53, 0.26 and 0.24 s, the isoflurane mean still below tolerance.

In [ ]:
mstate = pd.DataFrame(state_rows["magnitude"])
print(mstate.to_string(index=False))

**Result: partially reproduced.** Magnitude falls from quiet wakefulness to running and isoflurane in every reading, Friedman p <= 0.0004 and quiet the largest. With reading (a) at the medium current, unmatched, 17 deep animals, the means are 5.89, 1.51 and 1.23 (Q = 21.3, p = 2.4e-05): quiet and running are within tolerance, isoflurane is 0.07 below 1.6 +/- 0.3. At the high current quiet (8.96) and running (3.40) are above tolerance. Reading (b), with the area under |z| as divisor, is shown but does not decide the outcome (fig1f). With the baseline ending 20 ms before the pulse (variant, outside the criterion) the medium current gives 6.17, 1.56 and 1.37 (Q = 20.6, p = 3.4e-05), all within tolerance; the outcome on the criterion's baseline therefore turns on the artifact residue in the last 20 ms of the baseline.

## fig8b-pcist-awake-anesthesia

> PCIst was higher in the awake than in the isoflurane-anesthetized state in all subjects (45.5 +/- 3.6 against
> 16.5 +/- 2.4).

PCIst of the code the paper cites, on the ERP of the good electrodes at 2.5 kHz, trials matched within site between
awake (quiet and running) and anesthetized, by random subsampling or the first trials.

In [ ]:
def pc_frame(level, fig, how):
    rows = []
    for r in res:
        d = {s: v for (lv, fg, s, hw), v in r["pcist"].items() if lv == level and fg == fig and hw == how}
        if d:
            rows.append({"subject": r["subject"], "site": r["site"], "depth": r["depth"], **d})
    return pd.DataFrame(rows)

In [ ]:
rows = []
for lvl in claar.LEVELS:
    for how in ("random", "first") + (("random-excl20", "first-excl20") if lvl == "medium" else ()):
        d = pc_frame(lvl, "fig8b", how)
        tt = stats.ttest_rel(d.awake, d.isoflurane)
        rows.append({"intensity": lvl, "matching": how, "sites": len(d), "awake": mean_sem(d.awake), "isoflurane": mean_sem(d.isoflurane),
                     "awake > iso": f"{int((d.awake > d.isoflurane).sum())}/{len(d)}", "t": round(tt.statistic, 2), "p": f"{tt.pvalue:.3g}",
                     "within tol": bool(abs(d.awake.mean() - 45.5) <= 3.6 and abs(d.isoflurane.mean() - 16.5) <= 2.4)})
pc8b = pd.DataFrame(rows)
print(pc8b.to_string(index=False))
d = pc_frame("high", "fig8b", "random").merge(pc_frame("medium", "fig8b", "random"), on=["subject", "site", "depth"], suffixes=("_high", "_medium"))
print("\nper site, random matching:")
print(d.round(1).to_string(index=False))

**Result: reproduced.** At the medium current with random matching, PCIst is 43.26 +/- 3.72 awake and 14.93 +/- 2.41 under isoflurane, both within tolerance of 45.5 +/- 3.6 and 16.5 +/- 2.4, higher awake in 30 of 30 sites, paired t-test p = 9.8e-11; first-trial matching gives 43.47 and 14.84. At the high current the anesthetized mean (23.4 to 23.6) is above tolerance and one site of 30 is higher under isoflurane. The dandiset has 30 sites with valid trials, the paper 31. With the PCIst baseline ending 20 ms before the pulse (variant) the medium current gives 44.59 and 15.87 (random) and 44.85 and 15.75 (first), within tolerance, 30 of 30 sites higher awake.

## fig8c-pcist-depth

> In the awake state PCIst was significantly lower for superficial than for deep stimulation (35.6 +/- 4.8 against
> 52.7 +/- 4.4).

In [ ]:
rows = []
for lvl in claar.LEVELS:
    for how, fig in (("all", "fig8c"), ("random", "fig8b"), ("first", "fig8b")):
        d = pc_frame(lvl, fig, how)
        s, dp = d[d.depth == "superficial"].awake, d[d.depth == "deep"].awake
        tt = stats.ttest_ind(s, dp)
        rows.append({"intensity": lvl, "trials": "all awake" if how == "all" else f"matched ({how})", "superficial": mean_sem(s), "deep": mean_sem(dp),
                     "t": round(tt.statistic, 2), "p": f"{tt.pvalue:.3g}",
                     "within tol": bool(abs(s.mean() - 35.6) <= 4.8 and abs(dp.mean() - 52.7) <= 4.4),
                     "p<0.05, sup lower": bool(tt.pvalue < 0.05 and s.mean() < dp.mean())})
pc8c = pd.DataFrame(rows)
print(pc8c.to_string(index=False))

**Result: reproduced.** Under every reading superficial PCIst is lower than deep, within tolerance of 35.6 +/- 4.8 and 52.7 +/- 4.4: 32.34 +/- 4.84 against 52.51 +/- 3.69 at the high current with all awake trials (p = 0.0023), and 32.90 against 51.24 at the medium current (p = 0.016); n = 12 and 18.

## fig8d-pcist-three-states

> PCIst differed between all states; highest in quiet wakefulness (55.2 +/- 10.3), lower when running
> (39.9 +/- 7.7), and lowest under anesthesia (14.0 +/- 5.8), with every subject higher when running than under
> anesthesia.

Sites with at least 30 running trials at the intensity used; three states matched by random subsampling (the
criterion's method; the first trials are shown as a variant). One-way RM ANOVA (statsmodels `AnovaRM`), post hoc
paired t-tests with Benjamini-Hochberg.

In [ ]:
from statsmodels.stats.anova import AnovaRM
rows = []
for lvl in claar.LEVELS:
    for how in ("random", "first") + (("random-excl20", "first-excl20") if lvl == "medium" else ()):
        for per in ("site", "animal"):
            d = pc_frame(lvl, "fig8d", how)
            if per == "animal":
                d = d.groupby("subject")[["quiet", "running", "isoflurane"]].mean().reset_index().assign(site="mean")
            if len(d) < 3:
                rows.append({"intensity": lvl, "matching": how, "unit": per, "n": len(d)})
                continue
            d["unit"] = d.subject + d.site
            long = d.melt(id_vars=["unit"], value_vars=["quiet", "running", "isoflurane"], var_name="state", value_name="pcist")
            an = AnovaRM(long, "pcist", "unit", within=["state"]).fit().anova_table
            pairs = [("quiet", "running"), ("quiet", "isoflurane"), ("running", "isoflurane")]
            pp = [stats.ttest_rel(d[a], d[b]).pvalue for a, b in pairs]
            pc_ = claar.bh(pp)
            rows.append({"intensity": lvl, "matching": how, "unit": per, "n": len(d), "animals": d.subject.nunique(),
                         **{s: mean_sem(d[s]) for s in ("quiet", "running", "isoflurane")},
                         "F": f"F[{int(an['Num DF'].iloc[0])},{int(an['Den DF'].iloc[0])}]={an['F Value'].iloc[0]:.1f}", "p": f"{an['Pr > F'].iloc[0]:.3g}",
                         "post hoc q (q-r, q-i, r-i)": ", ".join(f"{x:.3g}" for x in pc_),
                         "running > iso": f"{int((d.running > d.isoflurane).sum())}/{len(d)}",
                         "within tol": bool(abs(d.quiet.mean() - 55.2) <= 10.3 and abs(d.running.mean() - 39.9) <= 7.7 and abs(d.isoflurane.mean() - 14.0) <= 5.8)})
pc8d = pd.DataFrame(rows)
print(pc8d.to_string(index=False))
print("\nsites with >= 30 running trials, high current:")
print(pc_frame("high", "fig8d", "random").round(1).to_string(index=False))
print("\nmatched trials per state and site (the smallest of quiet, running, isoflurane; the paper's average is 51):")
for lvl in claar.LEVELS:
    nm = [r["n_matched"][(lvl, "fig8d")] for r in res if (lvl, "fig8d") in r["n_matched"]]
    print(f"  {lvl}: {nm}, mean {np.mean(nm):.1f}")

**Result: partially reproduced.** With random matching (the criterion's method) the state effect is significant and the order is quiet > running > isoflurane in every row. At the high current, per site, the three means are within tolerance (45.65, 38.69, 15.01; F[2,24] = 14.9, p = 6e-05; running above isoflurane in 13 of 13), but quiet against running is not significant after correction (q = 0.24); per animal (11) the same holds (q = 0.34). At the medium current the quiet against running pair has q = 0.057 per site and 0.095 per animal. Only first-trial matching at the medium current makes every pair significant with the means within tolerance (q = 0.021, 0.0009, 0.0018 per site; 8 animals, the paper's 8). Matched trials average 41 per state at the high current and 45 at the medium current, against the paper's 51. With the PCIst baseline ending 20 ms before the pulse (variant) at the medium current, random matching puts the three means within tolerance per site (45.19, 32.38, 10.20) with quiet against running q = 0.059 (0.098 per animal), and first-trial matching keeps every pair significant (q = 0.027, 0.0014, 0.0020 per site).

## fig8e-pcist-recovery

> After isoflurane was stopped, PCIst rose in all but one subject within the first 30 min (32.7 +/- 4.9 against
> 14.3 +/- 2.3 anesthetized and 48.0 +/- 4.0 awake) and did not increase further in the next 30 min (36.4 +/- 5.0).

Readings for the sample: (a) every site with a first recovery stage (means over them; the RM ANOVA on those with all
four stages); (b) only sites with all four stages. Trials matched across the stages of a site.

In [ ]:
rows = []
for lvl in claar.LEVELS:
    for how in ("random", "first") + (("random-excl20", "first-excl20") if lvl == "medium" else ()):
        d = pc_frame(lvl, "fig8e", how)
        full = d.dropna(subset=["recovery2"]) if "recovery2" in d else d.iloc[:0]
        full = full.assign(unit=full.subject + full.site)
        long = full.melt(id_vars=["unit"], value_vars=["awake", "isoflurane", "recovery1", "recovery2"], var_name="state", value_name="pcist")
        an = AnovaRM(long, "pcist", "unit", within=["state"]).fit().anova_table if len(full) >= 3 else None
        pairs = [("awake", "isoflurane"), ("awake", "recovery1"), ("awake", "recovery2"), ("isoflurane", "recovery1"), ("isoflurane", "recovery2"), ("recovery1", "recovery2")]
        q = claar.bh([stats.ttest_rel(full[a], full[b]).pvalue for a, b in pairs]) if len(full) >= 3 else [np.nan] * 6
        for sample, dd in (("a: all with recovery1", d), ("b: all four stages", full)):
            rows.append({"intensity": lvl, "matching": how, "sample": sample, "sites": len(dd),
                         **{s: mean_sem(dd[s].dropna()) for s in ("awake", "isoflurane", "recovery1", "recovery2")},
                         "rec1 > iso": f"{int((dd.recovery1 > dd.isoflurane).sum())}/{len(dd)}",
                         "RM ANOVA (four stages)": "" if an is None else f"F[{int(an['Num DF'].iloc[0])},{int(an['Den DF'].iloc[0])}]={an['F Value'].iloc[0]:.1f}, p={an['Pr > F'].iloc[0]:.3g}",
                         "q rec1-rec2": round(q[5], 3), "q awake-rec1": round(q[1], 3), "q iso-rec1": round(q[3], 3),
                         "within tol": bool(abs(dd.awake.mean() - 48.0) <= 4.0 and abs(dd.isoflurane.mean() - 14.3) <= 2.3 and abs(dd.recovery1.mean() - 32.7) <= 4.9 and abs(dd.recovery2.dropna().mean() - 36.4) <= 5.0)})
pc8e = pd.DataFrame(rows)
print(pc8e.to_string(index=False))
print("\nper site, high current, random matching:")
print(pc_frame("high", "fig8e", "random").round(1).to_string(index=False))
print("\nper site, high current, first trials matched:")
print(pc_frame("high", "fig8e", "first").round(1).to_string(index=False))
print("\nper site, medium current, random matching:")
print(pc_frame("medium", "fig8e", "random").round(1).to_string(index=False))
print("\nper site, medium current, first trials matched:")
print(pc_frame("medium", "fig8e", "first").round(1).to_string(index=False))
for lvl in claar.LEVELS:
    print(f"\nmatched trials per stage and site (fig8e), {lvl} current:",
          {f"{r['subject']} {r['site']}": r["n_matched"][(lvl, "fig8e")] for r in res if (lvl, "fig8e") in r["n_matched"]})
# a minimum of matched trials per stage (a check; the paper states none)
rows = []
for lvl in claar.LEVELS:
    ok = {(r["subject"], r["site"]) for r in res if r["n_matched"].get((lvl, "fig8e"), 0) >= 20}
    for how in ("random", "first"):
        d = pc_frame(lvl, "fig8e", how)
        d = d[[(a, b) in ok for a, b in zip(d.subject, d.site)]]
        full = d.dropna(subset=["recovery2"]) if "recovery2" in d else d.iloc[:0]
        for sample, dd in (("a: all with recovery1", d), ("b: all four stages", full)):
            rows.append({"intensity": lvl, "matching": how, "sample": sample, "sites": len(dd),
                         **{s: mean_sem(dd[s].dropna()) for s in ("awake", "isoflurane", "recovery1", "recovery2")},
                         "rec1 > iso": f"{int((dd.recovery1 > dd.isoflurane).sum())}/{len(dd)}",
                         "within tol": bool(len(dd) and abs(dd.awake.mean() - 48.0) <= 4.0 and abs(dd.isoflurane.mean() - 14.3) <= 2.3 and abs(dd.recovery1.mean() - 32.7) <= 4.9 and abs(dd.recovery2.dropna().mean() - 36.4) <= 5.0)})
print("\nsites with at least 20 matched trials per stage only:")
print(pd.DataFrame(rows).to_string(index=False))
print("\nERPs under isoflurane, high current, every site with a first recovery stage: largest |z| of the GFP from -20 to -2 ms, peak z 2 to 50 ms and 100 to 800 ms, against awake:")
rows = []
for r in res:
    if ("high", "fig8e") not in r["n_matched"]:
        continue
    t = r["times"]
    row = {"subject": r["subject"], "site": r["site"]}
    for st in ("awake", "isoflurane"):
        z = claar.gfp_z(r["erp"][("high", st)], t)
        row[f"{st} n"] = r["n"][("high", st)]
        row[f"{st} pre"] = round(float(np.abs(z[(t >= -0.020) & (t <= -0.002)]).max()), 1)
        row[f"{st} early"] = round(float(z[(t >= 0.002) & (t <= 0.050)].max()), 1)
        row[f"{st} late"] = round(float(z[(t >= 0.100) & (t <= 0.800)].max()), 1)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** At the medium current with the first trials matched, in the 8 sites with all four stages: awake 46.47, isoflurane 14.89, first recovery 30.74 and second recovery 34.06, all within tolerance; RM ANOVA F[3,21] = 16.9, p = 8.3e-06; first recovery above isoflurane in 8 of 8; recovery 2 against recovery 1 q = 0.475. In the 13 sites with a first recovery stage the means are also within tolerance but 2 of 13 sites do not rise. Random matching at the medium current puts the means of the 13 sites within tolerance, but the first recovery exceeds isoflurane in only 8 of 13. At the medium current every site has at least 23 matched trials per stage (546655 MOs-deep the fewest), so requiring 20 changes nothing; at the high current 546655 MOs-deep is matched on a single trial (isoflurane PCIst 306.6 with first trials), which drives the isoflurane mean of 49.67 +/- 21.78, and without it the high current still puts isoflurane above tolerance (26.98 to 38.78). The outcome rests on one combination of readings. With the PCIst baseline ending 20 ms before the pulse (variant, outside the criterion) that combination gives an isoflurane mean of 18.11, above 14.3 +/- 2.3, while the 13 sites with a first recovery stage (first trials) stay within tolerance (47.02, 15.16, 29.79, 36.52) with 10 of 13 rising.

## Summary

| Claim | Outcome |
|---|---|
| stim-depths | blocked: data missing (no stimulation depth in mm) |
| eeg-good-electrodes | reproduced |
| fig1d-two-erp-peaks | not reproduced |
| fig1-supp1-example | reproduced (medium current) |
| fig1e-erp-duration | partially reproduced |
| fig1f-erp-magnitude | reproduced (medium current) |
| fig1-supp2-first-peak-flips | blocked: data missing (no stimulation coordinates) |
| state-trial-counts | reproduced |
| fig7c-erp-duration-state | partially reproduced |
| fig7d-erp-magnitude-state | partially reproduced |
| fig8b-pcist-awake-anesthesia | reproduced (medium current) |
| fig8c-pcist-depth | reproduced |
| fig8d-pcist-three-states | partially reproduced |
| fig8e-pcist-recovery | reproduced (one reading) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's: the paper does not say which of the three currents it analysed, so the
highest and the middle current are both run; the middle current with all awake trials gives the paper's example
(1.011 s and 11.133 SD.s against 1.01 and 11.13) and the paper's p-values for Figures 1E and 1F (0.081 and 0.035
against 0.0843 and 0.0354), which supports it as the paper's choice. Good EEG electrodes are the file's
`is_data_valid` marks; trials the file marks invalid are left out, which removes one superficial site (586468, SSp)
entirely; the stimulus artifact peaks one sample before the first sample at or after the onset, so the 2 ms mask
starts one sample earlier than the onset sample; the 0.1 to 100 Hz filter is zero phase; the magnitude's division by
"the area under the baseline" cannot be computed (the signed area of a z-scored baseline is zero), so the magnitude
is the area above z = 3; PCIst uses the cited code with an exact, memory-saving count in its threshold search
(checked against the original above); recovery stages are split at 30 and 60 min after the end of the isoflurane
epoch.

The zero-phase filter spreads a residue of the stimulus artifact into the last 20 ms before the pulse (|z| of the
GFP above 10 in 12 of 30 sites), inside the z-score and PCIst baselines. With baselines that end 20 ms before the
pulse (a variant outside the criteria), fig1e-erp-duration and fig7d-erp-magnitude-state would have all means
within tolerance at the medium current, and the one passing reading of fig8e-pcist-recovery would not;
fig1f-erp-magnitude, fig8b-pcist-awake-anesthesia and fig8d-pcist-three-states would keep their results. The
outcomes stay on the criteria's baselines.

What was found about the dataset: 23 subjects and 31 stimulation sites, as in the paper; the trial table gives the
current, target, depth label, state and validity of every pulse, and the EEG electrode table marks good electrodes
(mean 23 per subject, as in the paper); the stimulation depth in mm and the coordinates of the stimulating
electrode are not in the files; all 720 pulses of one site are marked invalid without saying why.

In [ ]:
print(f"{(time.time() - started) / 60:.1f} min")